# 89. The Single vs. Dual Sourcing Problem

## Tier 1: The Pen & Paper Method (Multi-Objective Mathematical Formulation)

### Key assumptions
- Multiple components with varying criticality levels
- Multiple suppliers with different cost, reliability, and quality characteristics
- Single vs. dual sourcing decisions for each component
- Multi-period planning horizon with demand fluctuations
- Trade-off between cost minimization and supply chain reliability

### Approach (step-by-step)
1. **Problem Formulation**: Define sets, parameters, and decision variables
2. **Multi-Objective Optimization**: Set up cost and reliability objectives
3. **Constraint Definition**: Demand satisfaction, supplier selection, and quality requirements
4. **Solution Method**: Use weighted sum approach for multi-objective optimization
5. **Analysis**: Compare single vs. dual sourcing strategies

### What to look for in the results
- Optimal sourcing allocation between single and dual sourcing
- Trade-off analysis between cost and reliability
- Sensitivity of solutions to parameter changes
- Component-specific sourcing recommendations

### Concrete example (from the source)
We'll implement the MediTech Manufacturing scenario with:
- 2 components (Critical and Standard)
- 3 potential suppliers (A, B, C)
- 4-quarter planning horizon
- Multi-objective optimization balancing cost and reliability

In [1]:
from typing import Tuple, List, Dict, Set

# Import required libraries for mathematical optimization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
@dataclass
class SourcingData:
    """Data structure for sourcing problem parameters"""
    components: List[str]
    suppliers: List[str]
    periods: List[int]
    demand: Dict[Tuple[str, int], float]  # (component, period) -> demand
    costs: Dict[Tuple[str, str], float]   # (component, supplier) -> unit cost
    reliability: Dict[str, float]         # supplier -> reliability score
    quality: Dict[Tuple[str, str], float] # (component, supplier) -> quality score
    min_quality: float                     # minimum quality requirement

# Define the concrete example from the source
def create_concrete_example():
    """Create the MediTech Manufacturing example data"""
    
    components = ['Component_1_Critical', 'Component_2_Standard']
    suppliers = ['Supplier_A', 'Supplier_B', 'Supplier_C']
    periods = [1, 2, 3, 4]  # 4 quarters
    
    # Demand data from source: Component 1: [100, 120, 110, 130], Component 2: [200, 180, 220, 210]
    demand = {
        ('Component_1_Critical', 1): 100, ('Component_1_Critical', 2): 120,
        ('Component_1_Critical', 3): 110, ('Component_1_Critical', 4): 130,
        ('Component_2_Standard', 1): 200, ('Component_2_Standard', 2): 180,
        ('Component_2_Standard', 3): 220, ('Component_2_Standard', 4): 210
    }
    
    # Cost data from source table
    costs = {
        ('Component_1_Critical', 'Supplier_A'): 15,
        ('Component_1_Critical', 'Supplier_B'): 18,
        ('Component_1_Critical', 'Supplier_C'): 16,
        ('Component_2_Standard', 'Supplier_A'): 8,
        ('Component_2_Standard', 'Supplier_B'): 7,
        ('Component_2_Standard', 'Supplier_C'): 9
    }
    
    # Reliability scores from source
    reliability = {
        'Supplier_A': 0.85,
        'Supplier_B': 0.92,
        'Supplier_C': 0.88
    }
    
    # Quality scores (assumed based on reliability pattern)
    quality = {
        ('Component_1_Critical', 'Supplier_A'): 0.87,
        ('Component_1_Critical', 'Supplier_B'): 0.94,
        ('Component_1_Critical', 'Supplier_C'): 0.90,
        ('Component_2_Standard', 'Supplier_A'): 0.86,
        ('Component_2_Standard', 'Supplier_B'): 0.91,
        ('Component_2_Standard', 'Supplier_C'): 0.89
    }
    
    return SourcingData(
        components=components,
        suppliers=suppliers,
        periods=periods,
        demand=demand,
        costs=costs,
        reliability=reliability,
        quality=quality,
        min_quality=0.85
    )

# Create the instance
data = create_concrete_example()
print(f"Problem setup: {len(data.components)} components, {len(data.suppliers)} suppliers, {len(data.periods)} periods")

Problem setup: 2 components, 3 suppliers, 4 periods


In [ ]:
class MultiObjectiveSourcingOptimizer:
    """Multi-objective optimizer for single vs. dual sourcing decisions"""
    
    def __init__(self, data: SourcingData, cost_weight: float = 0.6, reliability_weight: float = 0.4):
        self.data = data
        self.cost_weight = cost_weight
        self.reliability_weight = reliability_weight
        
    def calculate_total_cost(self, sourcing_decisions: Dict) -> float:
        """Calculate total cost for given sourcing decisions"""
        total_cost = 0
        for component in self.data.components:
            for period in self.data.periods:
                demand_qty = self.data.demand[(component, period)]
                for supplier in self.data.suppliers:
                    allocation = sourcing_decisions.get((component, supplier), 0)
                    unit_cost = self.data.costs[(component, supplier)]
                    total_cost += allocation * demand_qty * unit_cost
        return total_cost
    
    def calculate_reliability_score(self, sourcing_decisions: Dict) -> float:
        """Calculate weighted reliability score"""
        total_reliability = 0
        for component in self.data.components:
            for supplier in self.data.suppliers:
                allocation = sourcing_decisions.get((component, supplier), 0)
                supplier_reliability = self.data.reliability[supplier]
                total_reliability += allocation * supplier_reliability
        return total_reliability / len(self.data.components)  # Average across components
    
    def check_constraints(self, sourcing_decisions: Dict) -> bool:
        """Check if sourcing decisions satisfy all constraints"""
        # Check demand satisfaction and allocation constraints
        for component in self.data.components:
            for period in self.data.periods:
                total_allocation = sum(sourcing_decisions.get((component, s), 0) 
                                      for s in self.data.suppliers)
                # Allow small numerical tolerance
                if abs(total_allocation - 1.0) > 0.001:
                    return False
        
        # Check quality constraints
        for component in self.data.components:
            weighted_quality = 0
            for supplier in self.data.suppliers:
                allocation = sourcing_decisions.get((component, supplier), 0)
                quality_score = self.data.quality[(component, supplier)]
                weighted_quality += allocation * quality_score
            if weighted_quality < self.data.min_quality:
                return False
        
        return True
    
    def generate_sourcing_scenarios(self) -> List[Dict]:
        """Generate all possible sourcing scenarios (single and dual sourcing)"""
        scenarios = []
        
        for component in self.data.components:
            # Generate single sourcing options
            for supplier in self.data.suppliers:
                scenario = {}
                for comp in self.data.components:
                    if comp == component:
                        scenario[(comp, supplier)] = 1.0  # 100% from this supplier
                        for other_supplier in self.data.suppliers:
                            if other_supplier != supplier:
                                scenario[(comp, other_supplier)] = 0.0
                    else:
                        # For other components, default to single sourcing from cheapest reliable supplier
                        best_supplier = min(self.data.suppliers, 
                                         key=lambda s: self.data.costs[(comp, s)])
                        scenario[(comp, best_supplier)] = 1.0
                        for other_supplier in self.data.suppliers:
                            if other_supplier != best_supplier:
                                scenario[(comp, other_supplier)] = 0.0
                
                if self.check_constraints(scenario):
                    scenarios.append(scenario)
            
            # Generate dual sourcing options (50-50 split)
            for i, supplier1 in enumerate(self.data.suppliers):
                for supplier2 in self.data.suppliers[i+1:]:
                    scenario = {}
                    for comp in self.data.components:
                        if comp == component:
                            scenario[(comp, supplier1)] = 0.5
                            scenario[(comp, supplier2)] = 0.5
                            for other_supplier in self.data.suppliers:
                                if other_supplier not in [supplier1, supplier2]:
                                    scenario[(comp, other_supplier)] = 0.0
                        else:
                            # For other components, default to single sourcing
                            best_supplier = min(self.data.suppliers, 
                                             key=lambda s: self.data.costs[(comp, s)])
                            scenario[(comp, best_supplier)] = 1.0
                            for other_supplier in self.data.suppliers:
                                if other_supplier != best_supplier:
                                    scenario[(comp, other_supplier)] = 0.0
                    
                    if self.check_constraints(scenario):
                        scenarios.append(scenario)
        
        return scenarios
    
    def solve_multi_objective(self) -> Tuple[Dict, float, float]:
        """Solve the multi-objective optimization problem"""
        best_scenario = None
        best_objective = float('inf')
        best_cost = float('inf')
        best_reliability = 0
        
        scenarios = self.generate_sourcing_scenarios()
        
        for scenario in scenarios:
            cost = self.calculate_total_cost(scenario)
            reliability = self.calculate_reliability_score(scenario)
            
            # Multi-objective: minimize cost, maximize reliability
            # Normalize and combine objectives
            normalized_cost = cost / 10000  # Normalize by typical cost scale
            normalized_reliability = 1 - reliability  # Convert to minimization
            
            combined_objective = (self.cost_weight * normalized_cost + 
                                 self.reliability_weight * normalized_reliability)
            
            if combined_objective < best_objective:
                best_objective = combined_objective
                best_scenario = scenario
                best_cost = cost
                best_reliability = reliability
        
        return best_scenario, best_cost, best_reliability

In [ ]:
# Solve the multi-objective sourcing problem
optimizer = MultiObjectiveSourcingOptimizer(data, cost_weight=0.6, reliability_weight=0.4)
best_scenario, best_cost, best_reliability = optimizer.solve_multi_objective()

print("=== MULTI-OBJECTIVE SOURCING OPTIMIZATION RESULTS ===")
print(f"\nOptimal Total Cost: ${best_cost:,.2f}")
print(f"Optimal Reliability Score: {best_reliability:.3f}")

print("\n=== OPTIMAL SOURCING STRATEGY ===")
for component in data.components:
    print(f"\n{component}:")
    component_allocations = [(s, best_scenario.get((component, s), 0)) 
                              for s in data.suppliers 
                              if best_scenario.get((component, s), 0) > 0]
    
    if len(component_allocations) == 1:
        supplier, allocation = component_allocations[0]
        print(f"  Single Sourcing: 100% from {supplier}")
    else:
        print(f"  Dual Sourcing:")
        for supplier, allocation in component_allocations:
            print(f"    {allocation*100:.0f}% from {supplier}")

=== MULTI-OBJECTIVE SOURCING OPTIMIZATION RESULTS ===

Optimal Total Cost: $12,570.00
Optimal Reliability Score: 0.885

=== OPTIMAL SOURCING STRATEGY ===

Component_1_Critical:
  Single Sourcing: 100% from Supplier_A

Component_2_Standard:
  Single Sourcing: 100% from Supplier_B


In [ ]:
# Sensitivity analysis: Compare different weighting scenarios
def sensitivity_analysis():
    """Perform sensitivity analysis on objective weights"""
    
    weight_scenarios = [
        (0.8, 0.2),  # Cost-focused
        (0.6, 0.4),  # Balanced
        (0.4, 0.6),  # Reliability-focused
        (0.2, 0.8)   # Highly reliability-focused
    ]
    
    results = []
    
    for cost_w, rel_w in weight_scenarios:
        optimizer = MultiObjectiveSourcingOptimizer(data, cost_weight=cost_w, reliability_weight=rel_w)
        scenario, cost, reliability = optimizer.solve_multi_objective()
        
        # Determine sourcing strategy type
        dual_sourcing_count = 0
        for component in data.components:
            non_zero_allocations = sum(1 for s in data.suppliers 
                                     if scenario.get((component, s), 0) > 0)
            if non_zero_allocations > 1:
                dual_sourcing_count += 1
        
        results.append({
            'Cost_Weight': cost_w,
            'Reliability_Weight': rel_w,
            'Total_Cost': cost,
            'Reliability': reliability,
            'Dual_Sourcing_Components': dual_sourcing_count
        })
    
    return pd.DataFrame(results)

# Perform sensitivity analysis
sensitivity_results = sensitivity_analysis()
print("=== SENSITIVITY ANALYSIS ===")
print(sensitivity_results.to_string(index=False))

=== SENSITIVITY ANALYSIS ===
 Cost_Weight  Reliability_Weight  Total_Cost  Reliability  Dual_Sourcing_Components
         0.8                 0.2     12570.0        0.885                         0
         0.6                 0.4     12570.0        0.885                         0
         0.4                 0.6     12570.0        0.885                         0
         0.2                 0.8     13030.0        0.900                         0


In [ ]:
# Create comprehensive visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Multi-Objective Sourcing Optimization Analysis', fontsize=16, fontweight='bold')

# 1. Cost vs Reliability Trade-off
ax1 = axes[0, 0]
scatter = ax1.scatter(sensitivity_results['Total_Cost'], sensitivity_results['Reliability'], 
                     c=sensitivity_results['Cost_Weight'], cmap='viridis', s=100, alpha=0.7)
ax1.set_xlabel('Total Cost ($)')
ax1.set_ylabel('Reliability Score')
ax1.set_title('Cost vs. Reliability Trade-off')
ax1.grid(True, alpha=0.3)
plt.colorbar(scatter, ax=ax1, label='Cost Weight')

# Add annotations for each point
for i, row in sensitivity_results.iterrows():
    ax1.annotate(f"C:{row['Cost_Weight']:.1f}", 
                (row['Total_Cost'], row['Reliability']), 
                xytext=(5, 5), textcoords='offset points', fontsize=8)

# 2. Sourcing Strategy Comparison
ax2 = axes[0, 1]
strategy_counts = sensitivity_results['Dual_Sourcing_Components'].value_counts()
colors = ['lightcoral', 'lightblue']
ax2.pie(strategy_counts.values, labels=[f'{count} Components Dual Sourcing' for count in strategy_counts.index], 
        autopct='%1.1f%%', colors=colors, startangle=90)
ax2.set_title('Sourcing Strategy Distribution')

# 3. Cost Breakdown by Component
ax3 = axes[1, 0]
component_costs = []
component_labels = []

for component in data.components:
    total_component_cost = 0
    for period in data.periods:
        demand_qty = data.demand[(component, period)]
        for supplier in data.suppliers:
            allocation = best_scenario.get((component, supplier), 0)
            unit_cost = data.costs[(component, supplier)]
            total_component_cost += allocation * demand_qty * unit_cost
    
    component_costs.append(total_component_cost)
    component_labels.append(component.replace('_', ' ').title())

bars = ax3.bar(component_labels, component_costs, color=['skyblue', 'lightcoral'])
ax3.set_ylabel('Total Annual Cost ($)')
ax3.set_title('Cost Breakdown by Component')
ax3.tick_params(axis='x', rotation=45)

# Add value labels on bars
for bar, cost in zip(bars, component_costs):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
             f'${cost:,.0f}', ha='center', va='bottom', fontweight='bold')

# 4. Supplier Reliability and Allocation
ax4 = axes[1, 1]
supplier_data = []
for supplier in data.suppliers:
    total_allocation = 0
    for component in data.components:
        total_allocation += best_scenario.get((component, supplier), 0)
    
    supplier_data.append({
        'Supplier': supplier.replace('_', ' '),
        'Reliability': data.reliability[supplier],
        'Total_Allocation': total_allocation * 100  # Convert to percentage
    })

supplier_df = pd.DataFrame(supplier_data)
x_pos = np.arange(len(supplier_df))

# Create twin axes for reliability and allocation
ax4_twin = ax4.twinx()

bars1 = ax4.bar(x_pos - 0.2, supplier_df['Reliability'], 0.4, 
                label='Reliability Score', color='gold', alpha=0.8)
bars2 = ax4_twin.bar(x_pos + 0.2, supplier_df['Total_Allocation'], 0.4, 
                     label='Total Allocation %', color='lightgreen', alpha=0.8)

ax4.set_xlabel('Suppliers')
ax4.set_ylabel('Reliability Score', color='gold')
ax4_twin.set_ylabel('Total Allocation (%)', color='lightgreen')
ax4.set_title('Supplier Performance vs. Allocation')
ax4.set_xticks(x_pos)
ax4.set_xticklabels([s.replace('Supplier ', 'S') for s in supplier_df['Supplier']])
ax4.tick_params(axis='y', labelcolor='gold')
ax4_twin.tick_params(axis='y', labelcolor='lightgreen')

# Add legends
lines1, labels1 = ax4.get_legend_handles_labels()
lines2, labels2 = ax4_twin.get_legend_handles_labels()
ax4.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
# Additional analysis: Compare single vs. dual sourcing impact
def compare_sourcing_strategies():
    """Compare performance metrics between single and dual sourcing strategies"""
    
    # Find best single sourcing and best dual sourcing scenarios
    all_scenarios = optimizer.generate_sourcing_scenarios()
    
    single_sourcing_scenarios = []
    dual_sourcing_scenarios = []
    
    for scenario in all_scenarios:
        # Count dual sourcing components
        dual_count = 0
        for component in data.components:
            non_zero = sum(1 for s in data.suppliers if scenario.get((component, s), 0) > 0)
            if non_zero > 1:
                dual_count += 1
        
        if dual_count == 0:
            single_sourcing_scenarios.append(scenario)
        elif dual_count > 0:
            dual_sourcing_scenarios.append(scenario)
    
    # Evaluate best scenarios for each strategy
    def evaluate_scenario(scenario):
        cost = optimizer.calculate_total_cost(scenario)
        reliability = optimizer.calculate_reliability_score(scenario)
        return cost, reliability
    
    # Best single sourcing (minimum cost)
    best_single = min(single_sourcing_scenarios, key=lambda s: evaluate_scenario(s)[0])
    single_cost, single_reliability = evaluate_scenario(best_single)
    
    # Best dual sourcing (minimum combined objective)
    best_dual = min(dual_sourcing_scenarios, 
                   key=lambda s: optimizer.cost_weight * evaluate_scenario(s)[0]/10000 + 
                              optimizer.reliability_weight * (1 - evaluate_scenario(s)[1]))
    dual_cost, dual_reliability = evaluate_scenario(best_dual)
    
    # Create comparison table
    comparison = pd.DataFrame({
        'Strategy': ['Single Sourcing', 'Dual Sourcing (Optimal)'],
        'Total_Cost': [single_cost, dual_cost],
        'Reliability_Score': [single_reliability, dual_reliability],
        'Cost_Increase_vs_Single': [0, dual_cost - single_cost],
        'Reliability_Improvement_vs_Single': [0, dual_reliability - single_reliability]
    })
    
    return comparison, best_single, best_dual

# Perform comparison
strategy_comparison, best_single_scenario, best_dual_scenario = compare_sourcing_strategies()

print("=== SINGLE vs. DUAL SOURCING COMPARISON ===")
print(strategy_comparison.to_string(index=False))

print("\n=== DETAILED STRATEGY ANALYSIS ===")
print("\nBest Single Sourcing Strategy:")
for component in data.components:
    for supplier in data.suppliers:
        allocation = best_single_scenario.get((component, supplier), 0)
        if allocation > 0:
            print(f"  {component}: 100% from {supplier}")

print("\nBest Dual Sourcing Strategy:")
for component in data.components:
    allocations = [(s, best_dual_scenario.get((component, s), 0)) 
                   for s in data.suppliers if best_dual_scenario.get((component, s), 0) > 0]
    if len(allocations) == 1:
        supplier, allocation = allocations[0]
        print(f"  {component}: 100% from {supplier}")
    else:
        print(f"  {component}: Dual Sourcing")
        for supplier, allocation in allocations:
            print(f"    {allocation*100:.0f}% from {supplier}")

=== SINGLE vs. DUAL SOURCING COMPARISON ===
               Strategy  Total_Cost  Reliability_Score  Cost_Increase_vs_Single  Reliability_Improvement_vs_Single
        Single Sourcing     12570.0             0.8850                      0.0                             0.0000
Dual Sourcing (Optimal)     12800.0             0.8925                    230.0                             0.0075

=== DETAILED STRATEGY ANALYSIS ===

Best Single Sourcing Strategy:
  Component_1_Critical: 100% from Supplier_A
  Component_2_Standard: 100% from Supplier_B

Best Dual Sourcing Strategy:
  Component_1_Critical: Dual Sourcing
    50% from Supplier_A
    50% from Supplier_C
  Component_2_Standard: 100% from Supplier_B


### Why this Tier exists vs earlier Tiers
This Tier 1 provides the mathematical foundation for the single vs. dual sourcing problem by:
- **Establishing the theoretical framework**: Multi-objective optimization formulation that captures the fundamental trade-offs
- **Providing benchmark solutions**: Exact optimization results that serve as reference points for heuristic methods
- **Enabling sensitivity analysis**: Systematic exploration of how objective weights affect optimal decisions
- **Quantifying trade-offs**: Precise measurement of cost vs. reliability implications

### Pros / Cons vs this approach
**Pros:**
- **Optimal solutions**: Guarantees mathematically optimal decisions within the defined model
- **Comprehensive analysis**: Captures multiple objectives and constraints systematically
- **Reproducible results**: Same inputs always produce same optimal solutions
- **Theoretical foundation**: Provides basis for understanding problem structure

**Cons:**
- **Computational complexity**: Exponential growth in solution space with more components/suppliers
- **Data requirements**: Needs precise parameter estimates for all costs and reliabilities
- **Static assumptions**: Assumes parameters are known and constant over time
- **Limited scalability**: Becomes impractical for large-scale problems

### When to use this Tier
- **Strategic planning**: High-stakes sourcing decisions where optimality is critical
- **Benchmark setting**: Establishing performance targets for heuristic methods
- **Policy analysis**: Understanding how different objective weights affect decisions
- **Small to medium problems**: Where computational resources are not a constraint
- **Academic research**: Developing theoretical understanding of sourcing trade-offs